# 01 · Explorar un PDF real antes de indexarlo

**Objetivo:** identificar qué partes del documento son texto, tablas, estructura o
gráficos; medir desacuerdos entre extractores y preparar una revisión manual de
dificultades. Este notebook es independiente de 00 y no requiere red, LLM ni embeddings.

Corpus predeterminado: **PI-RADS v2.1 (2019), ACR–ESUR–AdMeTech**, aportado por
el usuario con cobertura de uso confirmada. ESUR-2012 permanece disponible como
antecedente histórico, con licencia CC Attribution Noncommercial según el editor.
El catálogo conserva atribución, procedencia y política de cada documento.
Para comparar, seleccionar `DOCUMENT_ID = "esur-2012"` y ejecutar todo otra vez.

**Resultado:** perfil por página, visualizaciones, comparación de texto/orden,
candidatos de tabla/cabeceras y registros con procedencia para notebook 02.

## 1. Fundamento técnico: PDF no es una representación semántica

Un PDF puede almacenar glifos en coordenadas sin párrafos, encabezados ni tablas
explícitos. El orden del stream gráfico, el orden geométrico y el orden que lee una
persona pueden ser distintos. Una tabla se puede extraer como una lista de palabras
que pierde su estructura lógica. Las figuras pueden ser vectores, no imágenes
incrustadas; contar imágenes no cuenta figuras.

Compararemos **pypdf** (texto y layout) y **pdfplumber** (texto + geometría/tablas).
No hay OCR en esta fase. Una página casi vacía en extracción se revisa antes de
asumir que es escaneada. Ninguno de los extractores constituye la verdad de referencia.

Dos diagnósticos, no métricas de exactitud:

$$J(A,B)=|A\cap B|/|A\cup B|$$

sobre conjuntos de tokens (ignora orden y repeticiones), y ratio de coincidencia de
secuencias de tokens (sensible al orden y a la heurística de `SequenceMatcher`).
Un Jaccard alto puede coexistir con una tabla desordenada. Un desacuerdo bajo puede
significar que ambos parsers cometen el mismo error.

In [ ]:
import json
import platform
import re
import time
from collections import Counter
from datetime import UTC, datetime
from difflib import SequenceMatcher
from importlib.metadata import version

import matplotlib.pyplot as plt
import pandas as pd
import pdfplumber
import pypdfium2 as pdfium
from IPython.display import Markdown, display
from pypdf import PdfReader
from radiology_rag.ingestion.catalog import load_catalog, sha256_file
from radiology_rag.paths import project_root, within

ROOT = project_root()
LOW_TEXT_THRESHOLD = 100  # Heurística para revisión, no umbral universal de OCR.
catalog = load_catalog(ROOT / "data/catalog/documents.v1.json")
DOCUMENT_ID = catalog.get("active_document_id", "esur-2012")
doc = next(d for d in catalog["documents"] if d["id"] == DOCUMENT_ID)
PDF_PATH = within(ROOT, doc["local_path"])
if not PDF_PATH.is_file():
    raise FileNotFoundError(
        f"Falta {DOCUMENT_ID}. Incorporar la fuente y su procedencia; ver docs/sources.md."
    )
manifest_path = PDF_PATH.with_suffix(".source.json")
if not manifest_path.is_file():
    raise FileNotFoundError("Falta el manifiesto .source.json: registrar procedencia primero.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
source_hash = sha256_file(PDF_PATH)
assert source_hash == manifest["sha256"], "El PDF cambió respecto al manifiesto"
assert manifest["document_id"] == DOCUMENT_ID
assert manifest["version"] == doc["version"]
OUT = ROOT / "data/processed" / DOCUMENT_ID
OUT.mkdir(parents=True, exist_ok=True)
display(pd.Series({
    "título": doc["title"], "versión": doc["version"], "papel": doc["role"],
    "fuente": doc["source_url"], "licencia": doc["rights"]["license"],
    "sha256": source_hash, "bytes": PDF_PATH.stat().st_size,
}))
layout = json.loads(within(ROOT, doc["processing"]["layout_path"]).read_text(encoding="utf-8"))
printed_labels = {p["page_number"]: p.get("page_label") for p in layout["pages"]}

def informative_tables(candidates):
    # El encabezado decorativo de PI-RADS también produce una falsa rejilla.
    bands = layout.get("decorative_image_bands", [])
    return [t for t in candidates if not any(
        a <= (t.bbox[0] + t.bbox[2])/2 <= c and
        b <= (t.bbox[1] + t.bbox[3])/2 <= d for a, b, c, d in bands)]


## 2. Inventario: metadatos, páginas y marcadores

La página física se numera desde 1; `pdf_page_label` procede del PDF y puede ser una
etiqueta automática, no el número editorial impreso. Comprobarlo al renderizar.
Los marcadores tampoco garantizan una jerarquía completa. No confundir metadatos
del productor del PDF con autoría científica.
`page_label` usa la numeración impresa revisada en el mapa cuando está disponible.

In [ ]:
reader = PdfReader(PDF_PATH)
assert not reader.is_encrypted, "No se procesan PDFs cifrados automáticamente"
print("Páginas físicas:", len(reader.pages))
display(pd.Series({str(k): str(v) for k, v in (reader.metadata or {}).items()}, name="metadata PDF"))

def outline_rows(items, depth=0):
    rows = []
    for item in items:
        if isinstance(item, list):
            rows.extend(outline_rows(item, depth + 1))
        else:
            page_index = reader.get_destination_page_number(item)
            rows.append({"depth": depth, "title": item.title,
                         "physical_page": page_index + 1 if page_index is not None else None})
    return rows

outline = pd.DataFrame(outline_rows(reader.outline), columns=["depth", "title", "physical_page"])
display(outline if not outline.empty else "Sin marcadores: la estructura deberá recuperarse del contenido.")

## 3. Implementación visible: perfil de cada página

Mediremos caracteres, palabras, tamaño y objetos geométricos. `find_tables()` propone
rejillas según líneas/bordes; un candidato puede ser falso y una tabla sin líneas
puede pasar inadvertida. Los tiempos son observaciones de una ejecución local, sin
calentamiento ni repeticiones: no permiten declarar un parser universalmente más rápido.

In [ ]:
def tokens(text):
    return re.findall(r"\w+", text.casefold())

rows, texts = [], {}
with pdfplumber.open(PDF_PATH) as pdf:
    for index, page in enumerate(pdf.pages):
        start = time.perf_counter()
        pypdf_text = reader.pages[index].extract_text() or ""
        pypdf_s = time.perf_counter() - start
        start = time.perf_counter()
        plumber_text = page.extract_text() or ""
        plumber_s = time.perf_counter() - start
        table_candidates = informative_tables(page.find_tables())
        ta, tb = tokens(pypdf_text), tokens(plumber_text)
        a, b = set(ta), set(tb)
        jaccard = len(a & b) / len(a | b) if a | b else 1.0
        sequence_ratio = SequenceMatcher(None, ta, tb, autojunk=True).ratio()
        rows.append({
            "page_number": index + 1,
            "page_label": printed_labels.get(index + 1) or reader.page_labels[index],
            "pdf_page_label": reader.page_labels[index],
            "width_pt": float(page.width), "height_pt": float(page.height),
            "pypdf_chars": len(pypdf_text), "plumber_chars": len(plumber_text),
            "pypdf_words": len(ta), "plumber_words": len(tb),
            "embedded_images": len(page.images), "lines": len(page.lines),
            "rectangles": len(page.rects), "table_candidates": len(table_candidates),
            "table_text_mentions": len(re.findall(r"Table\s+\d+", pypdf_text)),
            "replacement_chars": pypdf_text.count("\ufffd"),
            "low_text": len(pypdf_text.strip()) < LOW_TEXT_THRESHOLD,
            "token_jaccard": jaccard, "sequence_ratio": sequence_ratio,
            "pypdf_seconds": pypdf_s, "plumber_seconds": plumber_s,
        })
        texts[index + 1] = {"pypdf": pypdf_text, "pdfplumber": plumber_text}
profile = pd.DataFrame(rows)
display(profile.round(3))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
profile.plot(x="page_number", y=["pypdf_chars", "plumber_chars"], ax=axes[0], marker=".")
axes[0].set_ylabel("Caracteres")
axes[0].set_title(f"{DOCUMENT_ID} · densidad textual y estructura por página física")
profile.plot(x="page_number", y=["table_candidates", "embedded_images"], ax=axes[1], marker="o")
axes[1].set_ylabel("Objetos/candidatos")
profile.plot(x="page_number", y=["token_jaccard", "sequence_ratio"], ax=axes[2], marker=".")
axes[2].set_ylabel("Coincidencia (no exactitud)")
axes[2].set_xlabel("Página física (1-based)")
axes[2].set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(OUT / "page_profile.png", dpi=140)
plt.show()

### Interpretar el perfil antes de limpiar texto

Localiza páginas con poca extracción, tablas y desacuerdo de orden. La longitud
de texto por sí sola no distingue prosa correcta, cabeceras duplicadas o columnas
intercaladas. No borrar pies/cabeceras automáticamente: una nota puede contener una
condición esencial. El gráfico anterior se calcula con el documento de esta ejecución.

In [ ]:
review_candidates = profile.sort_values(["sequence_ratio", "token_jaccard"]).head(5)
display(review_candidates[["page_number", "page_label", "sequence_ratio",
                           "token_jaccard", "table_candidates", "low_text"]])
REVIEW_PAGE = int(review_candidates.iloc[0]["page_number"])
if profile.table_candidates.max() > 0:
    TABLE_PAGE = int(profile.loc[profile.table_candidates.idxmax(), "page_number"])
else:
    # Las tablas sin bordes pueden no detectarse. Las menciones sólo orientan revisión.
    mentioned = profile.loc[profile.table_text_mentions > 0, "page_number"].tolist()
    TABLE_PAGE = REVIEW_PAGE if REVIEW_PAGE in mentioned else (mentioned[0] if mentioned else 1)
    print("No hay rejillas detectadas por líneas; seleccionar una mención de tabla para inspección.")
print("Página para revisar orden:", REVIEW_PAGE, "| candidata a tablas:", TABLE_PAGE)
print("Se pueden cambiar estas variables y repetir las celdas siguientes.")
inspection_page = next((p["page_number"] for p in layout["pages"] for r in p["regions"]
    if r.get("group_id") == doc["processing"]["inspection_group_id"]), None)
if inspection_page:
    TABLE_PAGE = inspection_page
    print("Tabla seleccionada desde el mapa revisado:", TABLE_PAGE)


## 4. Contrastar el original renderizado con tres extracciones

Se renderiza el PDF original con PDFium (alternativa a Poppler en este entorno
Windows). Revisar la página completa, el pie impreso y el sentido de lectura antes
de interpretar la salida. La imagen es evidencia visual, no una corrección automática.

In [ ]:
def render_page(page_number, scale=1.35):
    if not 1 <= page_number <= len(reader.pages):
        raise ValueError("Página fuera de rango")
    pdf = pdfium.PdfDocument(PDF_PATH)
    page = pdf[page_number - 1]
    bitmap = page.render(scale=scale)
    image = bitmap.to_pil().copy()
    bitmap.close()
    page.close()
    pdf.close()
    return image

preview_pages = list(dict.fromkeys([1, REVIEW_PAGE, TABLE_PAGE]))
fig, axes = plt.subplots(1, len(preview_pages), figsize=(7 * len(preview_pages), 10), squeeze=False)
for ax, page_number in zip(axes[0], preview_pages):
    image = render_page(page_number)
    image.save(OUT / f"page_{page_number:03d}.png")
    ax.imshow(image)
    ax.set_title(f"Página física {page_number} · etiqueta PDF {reader.page_labels[page_number - 1]}")
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
layout_text = reader.pages[REVIEW_PAGE - 1].extract_text(extraction_mode="layout") or ""
for label, text in [
    ("pypdf por defecto", texts[REVIEW_PAGE]["pypdf"]),
    ("pypdf layout", layout_text),
    ("pdfplumber por defecto", texts[REVIEW_PAGE]["pdfplumber"]),
]:
    print(f"\n--- {label} · página física {REVIEW_PAGE} ---")
    print(text[:1800])
print("Salida limitada a 1800 caracteres por vista; el PDF y pages.jsonl conservan la página completa.")

### Experimento de tolerancia de espacios

`pdfplumber` agrupa caracteres próximos. Una tolerancia absoluta demasiado grande
puede unir palabras cuando la fuente es pequeña; reducirla puede fragmentar otras.
Comparar el valor por defecto con `x_tolerance=1` y con un umbral relativo al tamaño
de fuente. No elegir sólo por coincidencia con pypdf: revisar el original.

In [ ]:
tolerance_rows = []
reference_tokens = set(tokens(texts[REVIEW_PAGE]["pypdf"]))
with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[REVIEW_PAGE - 1]
    for label, options in [
        ("default", {}), ("x_tolerance=1", {"x_tolerance": 1}),
        ("x_tolerance_ratio=0.1", {"x_tolerance_ratio": 0.1}),
    ]:
        extracted = page.extract_text(**options) or ""
        current_tokens = set(tokens(extracted))
        union = current_tokens | reference_tokens
        tolerance_rows.append({
            "config": label, "characters": len(extracted), "words": len(tokens(extracted)),
            "jaccard_vs_pypdf": len(current_tokens & reference_tokens) / len(union) if union else 1,
        })
        print(f"\n{label}: {extracted[:350]}")
display(pd.DataFrame(tolerance_rows))
pd.DataFrame(tolerance_rows).to_csv(OUT / "word_spacing_comparison.csv", index=False)

### Experimento de orden geométrico: separar dos columnas

Una división al 50% puede mejorar lectura de prosa y romper tablas que abarcan todo
el ancho. La implementamos explícitamente para que ese supuesto sea inspeccionable.
Comparar la primera región de texto con la imagen: ¿se ha unido un encabezado a la
columna equivocada? No adoptaremos esta heurística como parser definitivo.

In [ ]:
with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[REVIEW_PAGE - 1]
    midpoint = page.width / 2
    left = page.crop((0, 0, midpoint, page.height)).extract_text() or ""
    right = page.crop((midpoint, 0, page.width, page.height)).extract_text() or ""
    columns_text = left + "\n" + right
    words = page.extract_words()
    centers = [(word["x0"] + word["x1"]) / 2 for word in words]
plt.figure(figsize=(10, 3))
plt.hist(centers, bins=40)
plt.axvline(midpoint, color="red", linestyle="--", label="corte supuesto")
plt.xlabel("Centro horizontal de palabra (puntos PDF)")
plt.ylabel("Palabras")
plt.title(f"Distribución geométrica · página física {REVIEW_PAGE}")
plt.legend()
plt.tight_layout()
plt.show()
print(columns_text[:2200])

## 5. Tablas: detección de rejilla frente a relaciones semánticas

Compararemos estrategias de `pdfplumber`: líneas visibles frente a alineación de
texto. Más celdas no equivale a mejor tabla. Es necesario comprobar títulos, unidades,
combinaciones de celdas, notas y continuaciones. Una tabla que se extiende a otra
página exige vinculación, que se diseñará en 02.

In [ ]:
strategies = {
    "lines": {"vertical_strategy": "lines", "horizontal_strategy": "lines"},
    "text": {"vertical_strategy": "text", "horizontal_strategy": "text"},
    "text-tight": {"vertical_strategy": "text", "horizontal_strategy": "text",
                   "text_x_tolerance": 1},
}
table_results, table_summary = {}, []
with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[TABLE_PAGE - 1]
    for name, strategy in strategies.items():
        found = informative_tables(page.find_tables(table_settings=strategy))
        table_results[name] = [table.extract() for table in found]
        table_summary.append({
            "strategy": name, "candidates": len(found),
            "rows_total": sum(len(t) for t in table_results[name]),
            "max_columns": max((len(row) for t in table_results[name] for row in t), default=0),
        })
    # Visualización de bordes/celdas propuestos; no altera el original.
    annotated = page.to_image(resolution=110)
    annotated.debug_tablefinder(table_settings=strategies["lines"])
    annotated.save(OUT / f"table_debug_{TABLE_PAGE:03d}.png")
    display(annotated.annotated)
display(pd.DataFrame(table_summary))
pd.DataFrame(table_summary).to_csv(OUT / "table_comparison.csv", index=False)
for name, found in table_results.items():
    print(f"Estrategia {name}: primer candidato (máximo 10 filas)")
    if found:
        display(pd.DataFrame(found[0]).head(10))
    else:
        print("Sin candidatos. Inspeccionar la imagen; no equivale a ausencia de tablas.")

**Ejercicio:** elige una fila del candidato y localiza en la imagen cada valor y su
encabezado. Anota si la tabla contiene una nota fuera de la rejilla. No copies una
categoría a un dataset hasta confirmar su relación con la condición completa y la
versión. Este cuaderno mide estructura, no emite recomendaciones radiológicas.

## 6. Cabeceras, pies y posibles secciones

Repetición de texto cerca de los márgenes sugiere encabezados/pies. Normalizamos
dígitos para detectar números de página; eso puede colapsar líneas de contenido
distintas. El umbral de 50% es una hipótesis visible, no una regla de eliminación.
Los títulos por tamaño de fuente también son sólo candidatos.

In [ ]:
margin_pages, heading_candidates = {}, []
with pdfplumber.open(PDF_PATH) as pdf:
    for index, page in enumerate(pdf.pages):
        pieces = [
            page.crop((0, 0, page.width, page.height * 0.08)).extract_text() or "",
            page.crop((0, page.height * 0.92, page.width, page.height)).extract_text() or "",
        ]
        candidates = set()
        for line in "\n".join(pieces).splitlines():
            normalized = re.sub(r"\d+", "#", " ".join(line.split())).strip()
            if normalized:
                candidates.add(normalized)
        for line in candidates:
            margin_pages.setdefault(line, []).append(index + 1)
        sizes = Counter(round(char["size"], 1) for char in page.chars)
        body_size = sizes.most_common(1)[0][0] if sizes else 0
        for word in page.extract_words(extra_attrs=["size"]):
            if word["size"] > body_size * 1.2:
                heading_candidates.append({"page_number": index + 1, "word": word["text"],
                                           "size": round(word["size"], 1), "top": round(word["top"], 1)})
recurring = pd.DataFrame([
    {"normalized_line": line, "page_count": len(pages), "pages": pages}
    for line, pages in margin_pages.items() if len(pages) >= max(2, len(reader.pages) * 0.5)
], columns=["normalized_line", "page_count", "pages"])
display(recurring)
display(pd.DataFrame(heading_candidates).head(30))
print("No se ha eliminado texto ni asignado secciones automáticamente.")

## 7. Exportar evidencia técnica para 02

Guardar texto sólo localmente. Cada página conserva fuente, hash y versión, parser
y su versión; `section=null` expresa trabajo pendiente. Los diagnósticos no son
etiquetas de verdad. La plantilla de revisión deja observaciones vacías para que
una persona las complete; no se presenta como un gold standard.

In [ ]:
created_at = datetime.now(UTC).isoformat()
page_records = []
for page_number, parser_texts in texts.items():
    page_records.append({
        "document_id": DOCUMENT_ID, "document_version": doc["version"],
        "document_role": doc["role"], "source_url": doc["source_url"],
        "source_sha256": source_hash, "page_number": page_number,
        "page_label": printed_labels.get(page_number) or reader.page_labels[page_number - 1],
        "pdf_page_label": reader.page_labels[page_number - 1], "section": None,
        "text": parser_texts["pypdf"], "parser": "pypdf.default",
        "parser_version": version("pypdf"), "created_at": created_at,
    })
(OUT / "pages.jsonl").write_text(
    "".join(json.dumps(record, ensure_ascii=False) + "\n" for record in page_records),
    encoding="utf-8",
)
profile.to_csv(OUT / "page_profile.csv", index=False)
review_path = OUT / "manual_review.csv"
if not review_path.exists():
    pd.DataFrame([{
        "document_id": DOCUMENT_ID, "source_sha256": source_hash, "page_number": p,
        "region": "", "reading_order_ok": "", "table_relations_ok": "",
        "printed_page": "", "observed_problem": "", "reviewer": "", "status": "unreviewed",
    } for p in preview_pages]).to_csv(review_path, index=False)
summary = {
    "document_id": DOCUMENT_ID, "version": doc["version"], "source_sha256": source_hash,
    "created_at": created_at, "pages": len(profile),
    "total_pypdf_chars": int(profile.pypdf_chars.sum()),
    "total_plumber_chars": int(profile.plumber_chars.sum()),
    "low_text_pages": profile.loc[profile.low_text, "page_number"].tolist(),
    "table_candidates": int(profile.table_candidates.sum()),
    "embedded_images": int(profile.embedded_images.sum()),
    "mean_token_jaccard": float(profile.token_jaccard.mean()),
    "mean_sequence_ratio": float(profile.sequence_ratio.mean()),
    "selected_review_pages": preview_pages,
    "word_spacing_comparison": tolerance_rows,
    "table_strategy_comparison": table_summary,
    "environment": {"python": platform.python_version(), **{
        p: version(p) for p in ("pypdf", "pdfplumber", "pypdfium2", "pandas", "matplotlib")}},
    "semantic_accuracy_measured": False,
    "llm_call_performed": False,
}
(OUT / "exploration.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
assert len(page_records) == len(reader.pages)
assert all(r["source_sha256"] == source_hash and r["page_number"] >= 1 for r in page_records)
print("Exportado en", OUT)

In [ ]:
display(Markdown(
    f"### Observaciones calculadas de esta ejecución\n"
    f"- Documento: **{DOCUMENT_ID}**, versión **{doc['version']}**.\n"
    f"- {len(profile)} páginas; {int(profile.pypdf_chars.sum()):,} caracteres con pypdf.\n"
    f"- {int(profile.low_text.sum())} páginas bajo el umbral de {LOW_TEXT_THRESHOLD} caracteres.\n"
    f"- {int(profile.table_candidates.sum())} candidatos de tabla con estrategia de líneas.\n"
    f"- Jaccard medio {profile.token_jaccard.mean():.3f}; coincidencia secuencial media "
    f"{profile.sequence_ratio.mean():.3f}. **No son exactitud de extracción.**\n"
    f"- Páginas propuestas para revisión visual: {preview_pages}."
))

## 8. Conclusiones, límites y siguiente experimento

**Qué hemos aprendido:** inventariar el archivo, preservar identidad/versionado,
comparar extracciones y contrastarlas con imágenes completas. La salida anterior
informa sólo de mediciones ejecutadas. Los candidatos y discrepancias orientan la
revisión; no demuestran que un parser sea correcto o que otro sea mejor.

**Problemas que debemos revisar:** orden de lectura, uniones de palabras, títulos
fuera de celdas, notas y segmentación de tablas. Documentar los problemas observados
en `manual_review.csv`; si no se ha revisado una región, mantener `unreviewed`.
No inferir problemas específicos a partir de una puntuación aislada.

**Cuándo usar esta técnica:** antes de indexar un PDF nuevo o una nueva versión.
Es un control de calidad del corpus, todavía no un sistema de consulta clínica.
Ni el hash ni la inspección de tablas validan criterios médicos.

**02:** anotar 6–10 regiones representativas con referencia manual y comparar
extracción/chunking fijo, estructural y semántico. Medir cuántas condiciones y notas
quedan separadas de la evidencia. Mantener el mismo corpus y no transferir conclusiones
de ESUR-2012 al manual actual sin repetir los experimentos.

Ejercicios: cambiar `REVIEW_PAGE`; probar corte de columnas en una tabla de ancho
completo; cambiar tolerancias de `find_tables`; justificar cada cambio con una
región revisada. Las mejoras se afirmarán sólo tras esa comparación.

Documentación: [pypdf](https://pypdf.readthedocs.io/en/stable/user/extract-text.html),
[pdfplumber](https://github.com/jsvine/pdfplumber).